# 公司行動、漲跌停與上下市稽核（B3／B4／B5，離線，不需網路）

輸入：
- 官方參考資料 `reference/corporate_actions/v1`
- lake 已發布的 FinLab 原始日 K、FinLab 還原日 K、`fubon_5m`

這些都只讀，不修改。

步驟：
1. **事件表**：把官方表統一成事件表，包括除息、除權、減資、面額變更、ETF 分割與反分割；另建上市與下市日期表。
2. **B4 漲跌停**：
   - 用升降單位和漲跌幅比例，重算官方事件日的漲跌停價，回報一致率；
   - 一般交易日則以前一日收盤推算漲跌停，檢查 FinLab 的最高、最低價是否超出。
3. **B3**：
   - 官方「除權息前收盤」是否等於 FinLab 原始收盤；
   - 事件日的價格是否在官方漲跌停之內；
   - FinLab 還原因子的跳變日能否對到官方事件。
4. **B5**：官方上市與下市日期，對照 FinLab 第一個與最後一個價格日；列出 lake 推斷已下市但沒有官方下市紀錄的代號。
5. **v0.2 新增**：
   - 官方調整因子表：生效日順延到下一個交易日，取代 FinLab 還原價；
   - 說明 FinLab 調整日與官方事件的對應；
   - 逐筆說明漲跌停超出的可能原因；
   - 檢查轉板代號在品質遮罩中的交易所；
   - 追查 8101。
6. **5 分 K**（`WITH_5M=True`）：事件日的 5 分 K 是否落在官方漲跌停之內。

結果寫在新的資料夾，最後複製到 Drive 的 `audits/corporate_events/`。**依序執行全部格子即可**，約 5–15 分鐘。


In [ ]:
# 1. 參數
REPO_OWNER, REPO_NAME, BRANCH = "hh4832", "tw-market-data-lake", "main"
DATA_ROOT = "/content/drive/MyDrive/00Quant_Research/data"
DATASET_ROOT = f"{DATA_ROOT}/_tw-market-data-lake/reference/corporate_actions/v1"
ARCHIVE = f"{DATA_ROOT}/_tw-market-data-lake/audits/corporate_events"
PROJECT_DIR = f"/content/{REPO_NAME}"
OUT_DIR = "/content/tw_lake_corp_events"
WITH_5M = True
MASK = f"{DATA_ROOT}/_tw-market-data-lake/audits/quality_acceptance/quality_mask_20261030_014633_fbec159/quality_mask_symbol_day.parquet"
TRACE = "8101"                 # 逐筆追查的代號


In [ ]:
# 2. 掛載 Google Drive
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
# 3. clone / pull + 依賴
import subprocess, os
if os.path.exists(f"{PROJECT_DIR}/.git"):
    subprocess.run(["git", "-C", PROJECT_DIR, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", PROJECT_DIR, "checkout", "-B", BRANCH, f"origin/{BRANCH}"], check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, f"https://github.com/{REPO_OWNER}/{REPO_NAME}.git", PROJECT_DIR], check=True)
print("HEAD:", subprocess.check_output(["git", "-C", PROJECT_DIR, "rev-parse", "HEAD"], text=True).strip())
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml>=6", "tqdm>=4.66"], check=True)
def stream(args):
    env = dict(os.environ, PYTHONPATH=f"{PROJECT_DIR}/src", PYTHONUNBUFFERED="1")
    p = subprocess.Popen(args, cwd=PROJECT_DIR, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="")
    return p.wait()

def ensure_drive():
    """Mount Drive safely. If an earlier cell unmounted Drive and then wrote under /content/drive,
    those files are only on this runtime's disk: move them aside (never delete) and copy any audit
    folders that are missing on the real Drive."""
    import os, shutil, time
    from google.colab import drive
    if os.path.ismount("/content/drive"):
        return
    stray = None
    if os.path.isdir("/content/drive") and os.listdir("/content/drive"):
        stray = f"/content/drive_local_{int(time.time())}"
        shutil.move("/content/drive", stray)
        print("moved local (unsynced) files to", stray)
    drive.mount("/content/drive")
    if stray:
        src_root = f"{stray}/MyDrive/00Quant_Research/data/_tw-market-data-lake/audits"
        dst_root = f"{DATA_ROOT}/_tw-market-data-lake/audits"
        for dirpath, dirnames, filenames in os.walk(src_root):
            rel = os.path.relpath(dirpath, src_root)
            for d in list(dirnames):
                target = os.path.join(dst_root, rel, d)
                if not os.path.exists(target):
                    shutil.copytree(os.path.join(dirpath, d), target)
                    print("copied missing", target)
                    dirnames.remove(d)

In [ ]:
# 4. 執行稽核
args = [sys.executable, "-m", "tw_lake.audit.corporate_events", "--dataset-root", DATASET_ROOT,
        "--data-root", DATA_ROOT, "--out-dir", OUT_DIR,
        "--mask", MASK, "--trace", TRACE]
if WITH_5M:
    args.append("--with-5m")
print("exit", stream(args))


In [ ]:
# 5. 重點結果，並複製到 Drive（把這格輸出貼給 Claude）
import glob, json, os, shutil
import pandas as pd
ensure_drive()
pd.set_option("display.width", 250); pd.set_option("display.max_rows", 200)
d = sorted(glob.glob(f"{OUT_DIR}/corporate_events_*"))[-1]
print(json.dumps(json.load(open(f"{d}/summary.json", encoding="utf-8")), ensure_ascii=False, indent=1))
for f in ("limit_rule_match.csv", "ordinary_day_limit_check.csv"):
    if os.path.exists(f"{d}/{f}"):
        print("\n==", f); print(pd.read_csv(f"{d}/{f}").to_string(index=False))
v = f"{d}/ordinary_day_limit_violations.csv"
if os.path.exists(v):
    x = pd.read_csv(v)
    print("\n== ordinary-day limit violations:", len(x), "| by symbol (top 30)")
    print(x.groupby("symbol").size().sort_values(ascending=False).head(30).to_string())
for f in ("finlab_jumps_explained.csv", "board_transfer_mask_check.csv"):
    if os.path.exists(f"{d}/{f}"):
        x = pd.read_csv(f"{d}/{f}", dtype={"symbol": str})
        print("\n==", f); print(x["status"].value_counts().to_string() if "status" in x.columns else x.to_string(index=False))
ve = f"{d}/ordinary_day_limit_violations_explained.csv"
if os.path.exists(ve):
    x = pd.read_csv(ve, dtype={"symbol": str})
    print("\n== violations by likely cause"); print(x.groupby(["instrument", "likely_cause"]).size().to_string())
    print(x[x["likely_cause"] == "unexplained"][["symbol", "date", "prev_close", "close", "high", "low", "limit_up",
                                               "limit_down"]].head(40).to_string(index=False))
for f in ("trace_official_rows.csv", "trace_finlab_prices.csv"):
    if os.path.exists(f"{d}/{f}"):
        x = pd.read_csv(f"{d}/{f}", dtype={"symbol": str})
        print("\n==", f, len(x)); print(x.tail(15).to_string(index=False) if "close" in x.columns else x.to_string(index=False))
c = f"{d}/inferred_delisted_vs_official.csv"
if os.path.exists(c):
    x = pd.read_csv(c, dtype={"symbol": str})
    print("\n== lake-inferred delisted without official record (sample)")
    print(x[~x["official_delisting"]].groupby("code_class").head(10).to_string(index=False))
dest = f"{ARCHIVE}/{os.path.basename(d)}"
shutil.copytree(d, dest)
print("copied to", dest)
from google.colab import drive
drive.flush_and_unmount()
